# **Geração de Embeddings Semânticos, Busca por Contexto e Extração de Tags**

---

# 🧠 Entendendo o Conceito

## 1. O que são Embeddings Semânticos?
Ao contrário do Passo 1 (que conta a frequência de palavras para classificar), no Passo 2 usamos **Deep Learning (Transformers)** para ensinar a IA a **entender o significado e o contexto** do texto.
- O modelo converte o título e o resumo de cada artigo em um **vetor denso de 384 números** (coordenadas numéricas no espaço semântico).
- **Exemplo:** Frases com palavras diferentes, mas com o mesmo sentido (ex: *"como proteger banco de dados"* e *"segurança em armazenamento SQL"*), ficam próximas nesse espaço geométrico.

## 2. Como funciona a Busca Semântica e a Recomendação?
Utilizamos a **Similaridade de Cosseno** para medir a distância entre os vetores:
- **Busca Semântica:** A consulta do usuário é convertida em um vetor e comparada com todos os **5.963 artigos**, retornando os resultados mais relevantes pelo contexto, mesmo sem bater palavras exatas.
- **Artigos Relacionados:** O sistema recomenda outros conteúdos encontrando os vetores mais próximos do artigo que o usuário está lendo no momento.

## 3. Como funciona a Extração Inteligente de Tags (KeyBERT + MMR)?
Para exibir palavras-chave na tela para o usuário, utilizamos o **KeyBERT**:
- O algoritmo extrai os *n-grams* (expressões de 1 a 2 palavras) mais representativos de cada artigo.
- **Diversificação com MMR (*Maximal Marginal Relevance*):** Removemos termos redundantes ou repetidos (como singular/plural ou sinônimos diretos), garantindo **5 tags ricas e variadas** por artigo.

## 4. Integração com o Banco de Dados (pgvector)
Os vetores gerados são exportados em formato estruturado (`.json` e `.npy`) prontos para carga no **PostgreSQL** com a extensão **`pgvector`**, permitindo consultas vetoriais de alta performance em produção.

In [ ]:
# ==============================================================================
# PASSO 2: GERAÇÃO DE EMBEDDINGS SEMÂNTICOS, BUSCA E RECOMENDAÇÃO
# ==============================================================================

# 1. Instalação das bibliotecas necessárias
!pip install sentence-transformers keybert pandas numpy -q

import json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from keybert import KeyBERT

# ------------------------------------------------------------------------------
# 1. CARREGAMENTO DOS DADOS E DO MODELO DE EMBEDDINGS
# ------------------------------------------------------------------------------
print("1. Carregando a base de dados tratada...")
df = pd.read_json("dataset_techmind_evolution.json")

# Enriquecendo a entrada de texto (Título + Resumo)
df["texto_completo"] = df["titulo"].fillna("") + ". " + df["resumo_limpo"].fillna("")

print("2. Carregando o modelo de Deep Learning (all-MiniLM-L6-v2)...")
# Este modelo transforma qualquer texto em um vetor denso de 384 números
model_embedding = SentenceTransformer('all-MiniLM-L6-v2')

# ------------------------------------------------------------------------------
# 2. GERAÇÃO DOS EMBEDDINGS SEMÂNTICOS
# ------------------------------------------------------------------------------
print("3. Gerando os Embeddings Semânticos para os 5.963 artigos (isso leva ~30s)...")
textos = df["texto_completo"].tolist()

# Vetorização de todo o dataset
embeddings = model_embedding.encode(textos, show_progress_bar=True, convert_to_numpy=True)

print(f"✅ Matrix de Embeddings gerada com sucesso! Formato: {embeddings.shape}")
# Formato esperado: (5963, 384) -> 5963 artigos, cada um com 384 coordenadas semânticas

# ------------------------------------------------------------------------------
# 3. EXTRAÇÃO DE PALAVRAS-CHAVE (TAGS) COM KEYBERT
# ------------------------------------------------------------------------------
print("\n4. Extraindo Palavras-Chave (Tags) com KeyBERT...")
kw_model = KeyBERT(model=model_embedding)

def extrair_tags(texto, top_n=5):
    keywords = kw_model.extract_keywords(texto, keyphrase_ngram_range=(1, 2), stop_words='english', top_n=top_n)
    return [kw[0] for kw in keywords]

# Exemplo de extração de tags para o artigo 0
exemplo_tags = extrair_tags(df["texto_completo"].iloc[0])
print(f"Artigo Exemplo: {df['titulo'].iloc[0]}")
print(f"🏷️ Tags Extraídas: {exemplo_tags}")

# ------------------------------------------------------------------------------
# 4. TESTE PRÁTICO: BUSCA SEMÂNTICA
# ------------------------------------------------------------------------------
print("\n=== TESTE DE BUSCA SEMÂNTICA ===")
query = "como proteger banco de dados contra ataques e invasões"
print(f"🔎 Consulta do Usuário: '{query}'\n")

# Transforma a consulta do usuário em um vetor de 384 números
query_embedding = model_embedding.encode(query, convert_to_numpy=True)

# Calcula a Similaridade de Cosseno entre a busca e TODOS os artigos da base
similaridades = util.cos_sim(query_embedding, embeddings)[0].numpy()

# Pega os 3 artigos mais similares
top_k_indices = np.argsort(similaridades)[::-1][:3]

print("Top 3 Artigos Encontrados Semanticamente:")
for rank, idx in enumerate(top_k_indices, 1):
    print(f"{rank}º - [{df['categoria_projeto'].iloc[idx]}] {df['titulo'].iloc[idx]}")
    print(f"   Score de Similaridade: {similaridades[idx]:.4f}\n")

# ------------------------------------------------------------------------------
# 5. EXPORTAÇÃO DOS ARTEFATOS PARA A SQUAD DE BACKEND / BD
# ------------------------------------------------------------------------------
print("\n5. Exportando artefatos semânticos...")

# 1. Salvando a matriz de embeddings bruta (.npy) para carregamento rápido no Python
np.save("embeddings_techmind.npy", embeddings)

# 2. Criando o dataframe de exportação com todas as colunas + a coluna de vetores
df_export = df.copy()

# Converte a matriz NumPy em lista Python para ser serializável em JSON
df_export["embedding"] = embeddings.tolist()

# 3. Exportando o JSON estruturado mantendo os acentos em português (force_ascii=False)
df_export.to_json("artigos_com_embeddings.json", orient="records", force_ascii=False, indent=4)

print("✅ SUCESSO ABSOLUTO!")
print("📁 Arquivos gerados no Colab:")
print("   - 'embeddings_techmind.npy' (Matriz de vetores leve)")
print("   - 'artigos_com_embeddings.json' (Pronto para carga no PostgreSQL / pgvector)")

1. Carregando a base de dados tratada...
2. Carregando o modelo de Deep Learning (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

3. Gerando os Embeddings Semânticos para os 5.963 artigos (isso leva ~30s)...


Batches:   0%|          | 0/187 [00:00<?, ?it/s]

✅ Matrix de Embeddings gerada com sucesso! Formato: (5963, 384)

4. Extraindo Palavras-Chave (Tags) com KeyBERT...
Artigo Exemplo: Representation Topology Divergence: A Method for Comparing Neural Network Representations
🏷️ Tags Extraídas: ['comparing neural', 'representation similarity', 'network representations', 'representation topology', 'network representation']

=== TESTE DE BUSCA SEMÂNTICA ===
🔎 Consulta do Usuário: 'como proteger banco de dados contra ataques e invasões'

Top 3 Artigos Encontrados Semanticamente:
1º - [Banco de Dados] CV-Rules: Serializability Verification of Concurrency Control Protocols via Explicit Transaction Ordering
   Score de Similaridade: 0.2837

2º - [Cibersegurança] Cool, But What About Oracles? An Oracle-Based Perspective on Blockchain Integration in the Accounting Field
   Score de Similaridade: 0.2771

3º - [Backend] Digital financial services and open banking innovation: are banks becoming invisible?
   Score de Similaridade: 0.2738


5. Exportan

**Download Artefatos:**

In [ ]:
from google.colab import files

# Baixa os artefatos de Embeddings e Busca Semântica do Passo 2
print("Iniciando o download dos artefatos do Passo 2...")

files.download('embeddings_techmind.npy')
files.download('artigos_com_embeddings.json')

Iniciando o download dos artefatos do Passo 2...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>